# Load UN Sustainable Development Goals (SDG) Indicator Metadata

This notebook fetches UN SDG indicator metadata - covering poverty, health, education, climate, and more.

In [ ]:
%pip install requests tqdm --quiet

In [ ]:
# Configuration
CATALOG = "main_catalog"
SCHEMA = "dev"
TABLE_NAME = "unsdg_indicators"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

FRESH_START = False

In [ ]:
import requests
from tqdm import tqdm
from pyspark.sql.types import StructType, StructField, StringType

In [ ]:
schema = StructType([
    StructField("indicator_id", StringType(), False),
    StructField("indicator_name", StringType(), True),
    StructField("long_definition", StringType(), True),
    StructField("source_organization", StringType(), True),
    StructField("source", StringType(), True),
    StructField("topics", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("periodicity", StringType(), True),
    StructField("aggregation_method", StringType(), True),
    StructField("license_type", StringType(), True),
    StructField("embedding_text", StringType(), True),
])

# SDG Goal descriptions for topic mapping
SDG_GOALS = {
    "1": "No Poverty",
    "2": "Zero Hunger",
    "3": "Good Health and Well-being",
    "4": "Quality Education",
    "5": "Gender Equality",
    "6": "Clean Water and Sanitation",
    "7": "Affordable and Clean Energy",
    "8": "Decent Work and Economic Growth",
    "9": "Industry, Innovation and Infrastructure",
    "10": "Reduced Inequalities",
    "11": "Sustainable Cities and Communities",
    "12": "Responsible Consumption and Production",
    "13": "Climate Action",
    "14": "Life Below Water",
    "15": "Life on Land",
    "16": "Peace, Justice and Strong Institutions",
    "17": "Partnerships for the Goals",
}

In [ ]:
# Get existing indicator IDs or create table
existing_ids = set()

if FRESH_START:
    spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE_NAME}")
    print("Fresh start - dropped existing table")
else:
    try:
        existing_df = spark.sql(f"SELECT indicator_id FROM {FULL_TABLE_NAME}")
        existing_ids = set(row.indicator_id for row in existing_df.collect())
        print(f"Resuming - found {len(existing_ids)} existing records")
    except:
        print("Table doesn't exist yet, starting fresh")

if not existing_ids or FRESH_START:
    empty_df = spark.createDataFrame([], schema)
    empty_df.write \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .mode("overwrite") \
        .saveAsTable(FULL_TABLE_NAME)
    
    # Set table description
    spark.sql(f"""
        COMMENT ON TABLE {FULL_TABLE_NAME} IS 
        'UN Sustainable Development Goals indicator metadata covering poverty, health, education, climate, and other global development targets.'
    """)
    print(f"Created table {FULL_TABLE_NAME} with CDF enabled")

In [ ]:
# Fetch all UN SDG indicators
print("Fetching UN SDG indicators...")
resp = requests.get("https://unstats.un.org/sdgapi/v1/sdg/Indicator/List", timeout=60)
resp.raise_for_status()
all_indicators = resp.json()

# Filter out already loaded
indicators = [ind for ind in all_indicators if ind.get("code") not in existing_ids]

print(f"Total UN SDG indicators: {len(all_indicators)}")
print(f"Already loaded: {len(existing_ids)}")
print(f"Remaining to load: {len(indicators)}")

In [ ]:
# Load indicators
for ind in tqdm(indicators, desc="Loading UN SDG indicators"):
    try:
        indicator_id = ind.get("code", "")
        indicator_name = ind.get("description", "") or ""
        goal = ind.get("goal", "")
        
        # Get goal description for topic
        topic = SDG_GOALS.get(str(goal), f"SDG Goal {goal}")
        
        # Build long definition from series descriptions
        series_list = ind.get("series", []) or []
        series_descriptions = [s.get("description", "") for s in series_list if s.get("description")]
        long_definition = "; ".join(series_descriptions[:5])
        
        # Create embedding text
        embedding_text = f"{indicator_name}. {long_definition}".strip()
        if not embedding_text or embedding_text == ".":
            embedding_text = indicator_name or indicator_id

        record = [(
            indicator_id,
            indicator_name,
            long_definition,
            "United Nations Statistics Division",
            "UN Sustainable Development Goals",
            topic,
            "",
            "",
            "",
            "",
            embedding_text,
        )]

        row_df = spark.createDataFrame(record, schema)
        row_df.write.format("delta").mode("append").saveAsTable(FULL_TABLE_NAME)

    except Exception as e:
        print(f"Error loading {indicator_id}: {e}")
        continue

print("Done!")

In [ ]:
# Verify
count = spark.sql(f"SELECT COUNT(*) FROM {FULL_TABLE_NAME}").collect()[0][0]
print(f"Total UN SDG indicators in table: {count}")
display(spark.sql(f"SELECT * FROM {FULL_TABLE_NAME} LIMIT 5"))